# 03 · LLM extraction: оценка извлечения договорённостей (gold)

Транскрипты (02) -> LLM (Ollama, Instructor) -> commitments -> сверка с `expected_commitments`
и метрики по категориям. Требует: транскрипты (генерируются здесь), `ollama` + `LLM_*` в `.env`.

In [1]:
# 1. Импорты, настройки, пути
import json
import sys
from pathlib import Path

RESEARCH_DIR = Path.cwd()
SRC_DIR = RESEARCH_DIR.parent / "src"  # services/api/src
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Settings (.env) читаются относительно CWD — переходим в services/api, где конфиг находит корневой .env.
import os
os.chdir(RESEARCH_DIR.parent)

from app.ml.extraction.context import ConversationContext
from app.ml.stt.schemas import TranscriptSegment
from app.ml.extraction.llm import InstructorLLMExtractor
from app.ml.extraction.service import ExtractionService
from app.ml.extraction.validation import normalize
from app.config import get_settings
settings = get_settings()

DATA_DIR = RESEARCH_DIR / "data"
GOLD_DIR = DATA_DIR / "gold"
AUDIO_DIR = DATA_DIR / "audio"
TRANSCRIPT_DIR = DATA_DIR / "transcripts"
EVAL_DIR = DATA_DIR / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
print("звонков в manifest:", len(manifest))
print("LLM:", settings.llm_model, "@", settings.llm_base_url or "-")
print("LLM включён:", settings.llm_enabled)

звонков в manifest: 14
LLM: qwen2.5:7b @ http://localhost:11434/v1
LLM включён: True


In [2]:
# 2. Транскрипты: если нет data/transcripts/<id>.json — прогнать STT (T-one + pyannote)
# Пропускает id без аудио. Модели уже закэшированы; диаризация — CPU, ~10-30 c/звонок.

from app.ml.asr.tone import ToneASRClient
from app.ml.diarization.pyannote import PyannoteDiarizer
from app.ml.stt.pipeline import TranscribePipeline, save_transcript


def ensure_transcripts() -> list[dict]:
    # Синхронная проверка наличия транскриптов (генерация - ниже в async).
    return [row for row in manifest if (TRANSCRIPT_DIR / f"{row['id']}.json").exists()]


async def generate_missing_transcripts() -> None:

    asr = ToneASRClient(settings.asr_ws_url)
    diarizer = PyannoteDiarizer(settings.hf_token) if settings.hf_token else None
    pipeline = TranscribePipeline(asr, diarizer)
    for row in manifest:
        out_path = TRANSCRIPT_DIR / f"{row['id']}.json"
        if out_path.exists():
            continue
        audio = AUDIO_DIR / f"{row['id']}.wav"
        if not audio.exists():
            print("no_audio", row["id"], flush=True)
            continue
        transcript = await pipeline.transcribe_call(row["id"], audio)
        save_transcript(transcript, TRANSCRIPT_DIR)
        print("stt:", row["id"], f"-> {len(transcript.segments)} сегментов", flush=True)


await generate_missing_transcripts()
rows = ensure_transcripts()
print("готово транскриптов:", len(rows))

готово транскриптов: 14


In [3]:
# 3. Контексты для извлечения: gold (дата/tz/участники) + транскрипт (сегменты)
def build_context(row: dict) -> ConversationContext | None:
    gold_path = GOLD_DIR / f"{row['id']}.json"
    trans_path = TRANSCRIPT_DIR / f"{row['id']}.json"
    if not (gold_path.exists() and trans_path.exists()):
        return None
    gold = json.loads(gold_path.read_text(encoding="utf-8"))
    trans = json.loads(trans_path.read_text(encoding="utf-8"))
    return ConversationContext(
        conversation_id=row["id"],
        conversation_date=gold["conversation_date"],
        user_timezone=gold["user_timezone"],
        participants=gold["participants"],
        transcript=[TranscriptSegment(**seg) for seg in trans["segments"]],
    )


contexts = {row["id"]: build_context(row) for row in rows}
print("контекстов:", sum(c is not None for c in contexts.values()))
missing = [cid for cid, ctx in contexts.items() if ctx is None]
if missing:
    print("нет контекста (нет gold или транскрипта):", missing)

контекстов: 14


In [4]:
# 4. Экстрактор (Instructor + Ollama) и сервис (извлечение + валидация)
assert settings.llm_enabled, "Задай LLM_BASE_URL/LLM_MODEL в .env и запусти ollama"

extractor = InstructorLLMExtractor(
    api_key=settings.llm_api_key,
    base_url=settings.llm_base_url,
    model=settings.llm_model,
)
service = ExtractionService(extractor)
print("экстрактор готов:", settings.llm_model)

экстрактор готов: qwen2.5:7b


In [5]:
# 5. Извлечение договорённостей по всем звонкам
results: dict[str, dict] = {}
for cid, ctx in contexts.items():
    if ctx is None:
        continue
    commitments = await service.extract(ctx)
    results[cid] = {
        "context": ctx,
        "turns": [seg.model_dump(mode="json") for seg in ctx.transcript],
        "commitments": [c.model_dump(mode="json") for c in commitments],
        "expected": json.loads((GOLD_DIR / f"{cid}.json").read_text(encoding="utf-8"))[
            "expected_commitments"
        ],
    }
    statuses = {c["status"] for c in results[cid]["commitments"]}
    print(f"{cid}: извлечено={len(commitments)} статусы={sorted(statuses)}")
print("извлечено по звонкам:", len(results))

explicit_deadline_01: извлечено=1 статусы=['confirmed']


meeting_02: извлечено=1 статусы=['confirmed']


payment_03: извлечено=1 статусы=['confirmed']


ambiguous_04: извлечено=1 статусы=['proposed']


contradictory_05: извлечено=1 статусы=['confirmed']


...

meeting_office_10: извлечено=1 статусы=['confirmed']


multiple_11: извлечено=2 статусы=['confirmed']


open_question_12: извлечено=0 статусы=[]


proposal_13: извлечено=0 статусы=[]


delivery_payment_14: извлечено=1 статусы=['confirmed']
извлечено по звонкам: 14


In [6]:
# 6. Метрики vs gold: recall/precision, совпадения type/owner/due_date, quote-пересечение
# Сводки — macro-среднее по звонкам (пустые-ожидаемые не искажают итог).


def words(text: str) -> set[str]:
    return set(normalize(text).split())


def quote_jaccard(a: str, b: str) -> float:
    wa, wb = words(a), words(b)
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)


def match_pair(expected: dict, extracted: dict) -> float:
    score = quote_jaccard(
        expected.get("context_quote", ""), extracted.get("context_quote", "")
    )
    if expected.get("owner") == extracted.get("owner"):
        score += 0.25
    if expected.get("due_date") == extracted.get("due_date"):
        score += 0.25
    return score


def evaluate_call(exp: list[dict], ext: list[dict]) -> dict:
    used = [False] * len(ext)
    matched = 0
    owner_ok = type_ok = date_ok = 0
    for e in exp:
        best, best_i = 0.0, -1
        for i, x in enumerate(ext):
            if used[i]:
                continue
            s = match_pair(e, x)
            if s > best:
                best, best_i = s, i
        if best_i >= 0 and best >= 0.5:
            used[best_i] = True
            matched += 1
            x = ext[best_i]
            type_ok += e.get("type") == x.get("type")
            owner_ok += e.get("owner") == x.get("owner")
            date_ok += e.get("due_date") == x.get("due_date")
    recall = matched / len(exp) if exp else (1.0 if not ext else 0.0)
    precision = matched / len(ext) if ext else (1.0 if not exp else 0.0)
    return {
        "expected": len(exp),
        "extracted": len(ext),
        "matched": matched,
        "recall": recall,
        "precision": precision,
        "type_match": type_ok / matched if matched else None,
        "owner_match": owner_ok / matched if matched else None,
        "date_match": date_ok / matched if matched else None,
    }


from collections import defaultdict

per_call: dict[str, dict] = {}
per_cat: dict[str, list[dict]] = defaultdict(list)
for cid, res in results.items():
    ev = evaluate_call(res["expected"], res["commitments"])
    per_call[cid] = ev
    cat = next(r["category"] for r in manifest if r["id"] == cid)
    per_cat[cat].append(ev)


def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


print(f"{'call':26s} exp extr match recall prec  type  owner  date")
for cid, ev in per_call.items():

    def f(v):
        return f"{v:.2f}" if v is not None else " -  "

    print(
        f"{cid:26s} {ev['expected']:3d} {ev['extracted']:4d} {ev['matched']:5d} {ev['recall']:.2f} {ev['precision']:.2f}  {f(ev['type_match'])}   {f(ev['owner_match'])}   {f(ev['date_match'])}"
    )

print()
macro_r = mean([ev["recall"] for ev in per_call.values()])
macro_p = mean([ev["precision"] for ev in per_call.values()])
tot_exp = sum(ev["expected"] for ev in per_call.values())
tot_match = sum(ev["matched"] for ev in per_call.values())
tot_extr = sum(ev["extracted"] for ev in per_call.values())
print(f"ИТОГО (macro):   recall={macro_r:.2f} precision={macro_p:.2f}")
print(
    f"ИТОГО (pooled):  recall={tot_match / max(1, tot_exp):.2f} precision={tot_match / max(1, tot_extr):.2f} (exp={tot_exp}, extr={tot_extr}, match={tot_match})"
)
print()
print(f"{'category':24s} n   recall  precision")
for cat, evs in sorted(per_cat.items()):
    print(
        f"{cat:24s} {len(evs):2d}  {mean([e['recall'] for e in evs]):.2f}    {mean([e['precision'] for e in evs]):.2f}"
    )

call                       exp extr match recall prec  type  owner  date
explicit_deadline_01         1    1     1 1.00 1.00  0.00   1.00   0.00
meeting_02                   1    1     0 0.00 0.00   -      -      -  
payment_03                   1    1     1 1.00 1.00  1.00   1.00   1.00
ambiguous_04                 1    1     1 1.00 1.00  0.00   1.00   1.00
contradictory_05             1    1     0 0.00 0.00   -      -      -  
no_commitment_06             0    0     0 1.00 1.00   -      -      -  
follow_up_07                 1    1     1 1.00 1.00  1.00   1.00   0.00
payment_advance_08           1    1     1 1.00 1.00  1.00   0.00   0.00
deadline_month_end_09        1    1     1 1.00 1.00  1.00   1.00   1.00
meeting_office_10            1    1     1 1.00 1.00  1.00   0.00   0.00
multiple_11                  3    2     2 0.67 1.00  0.50   0.50   0.50
open_question_12             1    0     0 0.00 0.00   -      -      -  
proposal_13                  1    0     0 0.00 0.00   -      - 

In [7]:
# 7. Сохранение итогов eval
out = {
    cid: {**per_call[cid], "commitments": res["commitments"], "turns": res["turns"]}
    for cid, res in results.items()
}
path = EVAL_DIR / "extraction_results.json"
path.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding="utf-8")
print("Сохранено:", path)

Сохранено: /Users/sofiya/Projects/MTS_Hackathon/services/api/research/data/eval/extraction_results.json
